In [ ]:
import torch
import torch.nn as nn
import numpy as np


In [ ]:
def LoSpixels(image_cube, mean_center=True):
    '''
    Convert image data cube from [Nx, Ny, Nz] to [Nz, Npix] format.

    INPUTS:
    image_cube: input data cube in image form, with shape [Nx, Ny, Nz] where Nz
    mean_center: if True, do mean centering for each frequency slice after 
    '''

    # swap axes to [Nz, Nx, Ny] for converting to visibilities:
    image_cube = np.swapaxes(image_cube,0,1)
    image_cube = np.swapaxes(image_cube,0,2)

    # conver to LoS pixels format [Nz, Npix]
    axes = np.shape(image_cube)
    image_LoSpixels = np.reshape(image_cube,(axes[0], axes[1]*axes[2]))
    
    # mean center the data:
    if mean_center==True:
        nz = np.shape(image_LoSpixels)[0]
        for i in range(nz):
            image_LoSpixels[i] = image_LoSpixels[i] - np.mean(image_LoSpixels[i])
    
    return image_LoSpixels

In [ ]:
# function from gpr4im

def PCAclean(Input, N_FG=4):
    '''
    Takes input in [Nx,Ny,Nz] data cube form where Nz is number of redshift 
    (frequency) bins. N_FG is number of eigenmodes for PCA to remove
    '''
    
    # Collapse data cube to [Npix,Nz] structure:
    axes = np.shape(Input)
    Input = LoSpixels(Input, mean_center=True)
    Nz,Npix = np.shape(Input)
    
    # Obtain frequency covariance matrix for input data
    C = np.cov(Input)
    eigenval, eigenvec = np.linalg.eigh(C)
    eignumb = np.linspace(1,len(eigenval),len(eigenval))
    eigenval = eigenval[::-1] #Put largest eigenvals first
    A = eigenvec[:,Nz-N_FG:] # Mixing matrix
    S = np.dot(A.T,Input)
    
    # PCA Component Maps
    Recon_FG = np.dot(A,S)
    Residual = Input - Recon_FG
    Residual = np.swapaxes(Residual,0,1) #[Nz,Npix]->[Npix,Nz]
    Residual = np.reshape(Residual,(axes[0],axes[1],axes[2]))
    
    return Residual,eignumb,eigenval

In [ ]:
torch.mean(torch.ones((3,2)))

In [ ]:
def LoSpixels_torch(image_cube, mean_center=True):
    '''
    Convert image data cube from [Nx, Ny, Nz] to [Nz, Npix] format.

    INPUTS:
    image_cube: input data cube in image form, with shape [Nx, Ny, Nz] where Nz
    mean_center: if True, do mean centering for each frequency slice after 
    '''

    # swap axes to [Nz, Nx, Ny] for converting to visibilities:
    image_cube = torch.swapaxes(image_cube,0,1)
    image_cube = torch.swapaxes(image_cube,0,2)

    # conver to LoS pixels format [Nz, Npix]
    axes = image_cube.shape
    image_LoSpixels = torch.reshape(image_cube,(axes[0], axes[1]*axes[2]))
    
    # mean center the data:
    if mean_center==True:
        nz = image_LoSpixels.shape[0]
        for i in range(nz):
            image_LoSpixels[i] = image_LoSpixels[i] - torch.mean(image_LoSpixels[i])
    
    return image_LoSpixels

In [ ]:
def PCAclean_torch(Input, N_FG=4):
    '''
    Takes input in [Nx,Ny,Nz] data cube form where Nz is number of redshift 
    (frequency) bins. N_FG is number of eigenmodes for PCA to remove
    '''
    
    # Collapse data cube to [Npix,Nz] structure:
    axes = Input.shape
    Input = LoSpixels_torch(Input, mean_center=True)
    Nz,Npix = Input.shape
    
    # Obtain frequency covariance matrix for input data
    C = torch.cov(Input)
    eigenval, eigenvec = torch.linalg.eigh(C)
    eignumb = torch.linspace(1,len(eigenval),len(eigenval))
    eigenval = torch.flip(eigenval, dims=[0]) #Put largest eigenvals first
    A = eigenvec[:,Nz-N_FG:] # Mixing matrix
    S = torch.matmul(A.T,Input) # might need to change
    
    # PCA Component Maps
    Recon_FG = torch.matmul(A,S)
    Residual = Input - Recon_FG
    Residual = torch.swapaxes(Residual,0,1) #[Nz,Npix]->[Npix,Nz]
    Residual = torch.reshape(Residual,(axes[0],axes[1],axes[2]))
    
    return Residual,eignumb,eigenval

In [ ]:
torch.matmul

In [ ]:
np.dot(

In [ ]:
torch.tensor(obs_vis)

In [ ]:
pca_clean = PCAclean_torch(torch.tensor(obs_vis[0, ...]), N_FG=7)[0]

In [ ]:
import h5py, os
galaxy_path = "/data101/makinen/hirax_sims/more_baselines/galaxy_gaussian_pb/"
cosmo_path = "/data101/makinen/hirax_sims/cosmo_gaussian_pb/"

In [ ]:
from tqdm import tqdm

def get_cosmo_data(datadir = cosmo_path,
                   num_train=20,
                  savedir="/data101/makinen/hirax_sims/baseline_48/"):
    
    files = os.listdir(datadir)
    print("all files", len(files))
    cosmo = []
    params = []
    files = [f for f in files if "cosmo" in f]
    print("cosmo-only files", len(files))
    
    for i,f in tqdm(enumerate(files[:num_train]), position=0):
        
        cosmo.append(np.array(h5py.File(datadir + f, "r")['/vis/']).T[:, :, :])
        p = f[9:11]
        if p == "pl":
            p = 67

        params.append(float(p)) # get H0 value
        
    if savedir is not None:
        np.save(savedir + "cosmo_vary_H0_vis", cosmo)
        np.save(savedir + "cosmo_vary_H0_params", np.array(params))
        
    return np.array(cosmo), np.array(params)


def get_galaxy_data(datadir = galaxy_path,
                   num_train=20,
                  savedir="/data101/makinen/hirax_sims/baseline_48/"):
    
    files = os.listdir(datadir)
    print("all files", len(files))
    sims = []
    
    files = [f for f in files if "galaxy_md" in f]
    print("galaxy-only files", len(files))
    
    for i,f in tqdm(enumerate(files[:num_train]), position=0):
        #print(f)
        sims.append(np.array(h5py.File(datadir + f, "r")['/vis/']).T[:, :, :])
            
        
    if savedir:
        np.save(savedir + "galaxy_vis", np.array(sims))
        
    return np.array(sims)

In [ ]:
savedir="/data101/makinen/hirax_sims/baseline_48/"

cosmo_vis, cosmo_params = get_cosmo_data(num_train=10, savedir=savedir)

galaxy_vis = get_galaxy_data(num_train=10, savedir=savedir)


galaxy_mask = 1024

full_cosmo = cosmo_vis[:, :galaxy_mask, :, :]
full_gal = galaxy_vis[:, :galaxy_mask, :, :]
#full_obs = np.load(outdir + "obs_gauss.npy")[:, :galaxy_mask, :, :]

full_cosmo = np.transpose(full_cosmo, (0, 3, 1, 2))
full_gal = np.transpose(full_gal, (0, 3, 1, 2))

obs_vis = full_cosmo + full_gal

In [ ]:
obs_vis.shape

In [ ]:
N_FG = 14

pca_clean = PCAclean_torch(torch.tensor(obs_vis[0, ...].real), N_FG=N_FG)[0]

In [ ]:
full_cosmo.shape, pca_clean.shape, full_gal.shape

In [ ]:
torch.stack((torch.tensor(full_cosmo[0, ...].real), torch.tensor(full_cosmo[0, ...].imag)), dim=-1).shape

In [ ]:
# do saliency plots
import matplotlib.pyplot as plt

random_sim = 0
ibase = 2


plt.figure(figsize=(10,6))

#vmin = -2e-5
#vmax = 2e-5

plt.subplot(141)
pos = plt.imshow((full_cosmo[0, ibase,:512,:].real), origin = 'lower', cmap='coolwarm')
plt.colorbar(pos, fraction=0.046)
plt.title('True HI')

plt.subplot(142)
pos = plt.imshow((pca_clean.numpy()[ ibase,:512,:]).real, origin = 'lower', cmap='coolwarm')
plt.title('PCA-%d'%(N_FG))
cbar = plt.colorbar(pos, fraction=0.046)
#cbar.ax.set_ylabel(r'T [mK]', rotation=270, fontsize=18)

plt.subplot(143)
pos = plt.imshow((pca_clean.numpy()[ibase,:512,:].real - full_cosmo[0, ibase, :512,:].real),  origin = 'lower', cmap='coolwarm')
cbar = plt.colorbar(pos, fraction=0.046)
#cbar.ax.set_ylabel(r'T [mK]', rotation=270, fontsize=18)

plt.title('Residual (PCA - HI)')


plt.subplot(144)
pos = plt.imshow((full_gal[0, ibase,:512,:].real), origin = 'lower', cmap='coolwarm')
cbar = plt.colorbar(pos, fraction=0.046)
#cbar.ax.set_ylabel(r'T [mK]', rotation=270, fontsize=18)

plt.title('galaxy')
plt.tight_layout()

In [ ]:
full_cosmo.shape

In [ ]:
cosmo

In [ ]:
torch.nn.ConvTranspose3d(output_padding=)

In [ ]:
torch.flip(torch.linspace(1, 100, 100), dims=[0])

In [ ]:
pwd

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nets import *
block = BasicBlock(64, 128)
model = UNet3d(BasicBlock)

In [ ]:
block

In [ ]:
outs, x_pca = model(torch.tensor(inputs[:8, ...]))

In [ ]:
inputs.shape, outs.shape

In [ ]:
plt.imshow(outs.detach().numpy()[0, 0, 0, :, :])
plt.colorbar()

In [ ]:
x_pca.shape

In [ ]:
plt.imshow(x_pca.detach().numpy()[0, 0, :, 0, :])
plt.colorbar()

In [ ]:
full_cosmo.shape

In [ ]:
plt.imshow(full_cosmo.real[0, :, 0, :])
plt.colorbar()

In [ ]:
obs_vis.shape

In [ ]:
inputs = np.stack([obs_vis.real, obs_vis.imag], axis=-1).astype(np.float32)

split = 1024 / 128 # 8 chunks per sky simulation

inputs = np.array(np.split(inputs, split, axis=2))
inputs = np.transpose(np.concatenate(inputs), (0, 4, 2, 3, 1))

In [ ]:
inputs.shape

In [ ]:
model(torch.tensor(inputs[:2, ...]))